GDF-Net Teacher M2 (Stage-Wise LLRD and Multi-Scale Training) on ISIC-2018
Dataset: ISIC-2018 Task 1 (2,594 train / 100 validation / 1,000 test images).
Protocol: ConvFormer-S18 trained with stage-wise layer-wise learning rate decay and multi-scale batch resizing (352x352, 384x384, 416x416), serving as Teacher M2 in online ensemble distillation (Section 3.3 and Appendix C).


Stage-wise LLRD and multi-scale training with official test evaluation.


In [1]:
"""
========================================================================================
Phase 24: GDF-Net (ConvFormer-S18) — Proposal 3 Optimization on ISIC-2018
Architecture : Canonical GDF-Net (convformer_s18.sail_in22k_ft_in1k_384 + MS-FDF +
               MaskGuidedLGAG + CascadedMSCAMDecoderBlock, 29.50M Params)
Key Upgrades (Proposal 3):
  1. 3-Epoch Decoder Warmup (Backbone Frozen): Stabilizes randomly initialized MS-FDF,
     MaskGuidedLGAG, and CascadedMSCAM gradients before unfreezing ImageNet-22K weights.
  2. Layer-Wise Learning Rate Decay (LLRD, 5 Groups):
     - Stem & Stage 1 (low-level edges/textures) : lr = 1.5e-5
     - Stage 2 (mid-level patterns)              : lr = 2.5e-5
     - Stage 3 (part-level representations)      : lr = 4.0e-5
     - Stage 4 (high-level semantic context)     : lr = 6.0e-5
     - Decoder & Frequency/Gate Heads            : lr = 2.0e-4
  3. Multi-Scale Training ({320x320, 384x384, 448x448}) + FixRes (Final 15 Epochs @ 384x384):
     Forces scale-invariant boundary learning across tiny nevi and giant melanomas.
  4. WarmupModelEMA (decay=0.998 with dynamic ramp (1+u)/(10+u)): Prevents early random
     decoder weights from polluting the exponential moving average.
  5. 60-Epoch Single-Cycle Cosine Schedule + 5-Mode Official 1,000 Blind Test Evaluation:
     - Single-View (T=0.50, No TTA)
     - Calibrated Single-View (T=T_opt_sv tuned on 100 Official Val, No TTA)
     - 4-View Flip TTA (T=0.50)
     - Calibrated 4-View Flip TTA (T=T_opt_tta tuned on 100 Official Val)
     - Multi-Scale 6-View TTA ({384, 448} + Flips, T=T_opt_ms tuned on 100 Official Val)
========================================================================================
"""

import os
import sys
import glob
import time
import copy
import json
import random
import zipfile
import shutil
import subprocess
import numpy as np
import pandas as pd
from PIL import Image
import cv2

from scipy.ndimage import distance_transform_edt, binary_erosion

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import timm
import albumentations as A
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

SCRIPT_START_TIME = time.time()
MAX_RUNTIME_SECONDS = 39600  # 11.0 hours safety guard
BACKBONE_NAME = "convformer_s18.sail_in22k_ft_in1k_384"
MODEL_TAG = "ConvFormerS18_LLRD_MS"
TOTAL_EPOCHS = 60
FREEZE_WARMUP_EPOCHS = 3
MULTISCALE_END_EPOCH = 45  # Epochs 4..45 use {320, 384, 448}; Epochs 46..60 lock at 384x384 (FixRes)


# ====================================================================
# 1. Reproducibility & Seed
# ====================================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)


# ====================================================================
# 2. Clinical Dermoscopy Augmentation Suite (15-Case Synthesis)
# ====================================================================
class RandomVignettingTransform:
    """Simulates circular dermoscopic optical lens vignetting (dark ring/corners)."""
    def __init__(self, p=0.25, min_radius=0.50, max_radius=0.80):
        self.p = p
        self.min_radius = min_radius
        self.max_radius = max_radius

    def __call__(self, image):
        if random.random() > self.p:
            return image
        H, W = image.shape[:2]
        center_x = random.uniform(W * 0.42, W * 0.58)
        center_y = random.uniform(H * 0.42, H * 0.58)
        radius = random.uniform(self.min_radius, self.max_radius) * min(H, W)
        Y, X = np.ogrid[:H, :W]
        dist = np.sqrt((X - center_x) ** 2 + (Y - center_y) ** 2)
        sigma = radius * 0.18
        mask = 1.0 - 1.0 / (1.0 + np.exp(-(dist - radius) / max(sigma, 1.0)))
        mask = np.clip(mask, 0.08, 1.0)
        if image.ndim == 3:
            mask = mask[..., None]
        return (image * mask).astype(np.uint8)


class SyntheticHairTransform:
    """Simulates curved dark and light/gray hair strands occluding skin and lesion borders."""
    def __init__(self, p=0.30, max_hairs=5):
        self.p = p
        self.max_hairs = max_hairs

    def __call__(self, image):
        if random.random() > self.p:
            return image
        H, W = image.shape[:2]
        img_aug = image.copy()
        num_hairs = random.randint(1, self.max_hairs)
        for _ in range(num_hairs):
            x0, y0 = random.randint(0, W), random.randint(0, H)
            x1, y1 = random.randint(0, W), random.randint(0, H)
            ctrl_x, ctrl_y = random.randint(0, W), random.randint(0, H)
            points = []
            for t in np.linspace(0, 1, 30):
                px = int((1 - t) ** 2 * x0 + 2 * (1 - t) * t * ctrl_x + t ** 2 * x1)
                py = int((1 - t) ** 2 * y0 + 2 * (1 - t) * t * ctrl_y + t ** 2 * y1)
                points.append((px, py))
            pts = np.array(points, np.int32).reshape((-1, 1, 2))
            thickness = random.randint(1, 2)
            if random.random() < 0.8:
                c = random.randint(10, 45)
                color = (c, max(0, c - 5), max(0, c - 10))
            else:
                c = random.randint(180, 230)
                color = (c, c, c)
            cv2.polylines(img_aug, [pts], isClosed=False, color=color, thickness=thickness, lineType=cv2.LINE_AA)
        return img_aug


class SyntheticRulerMarkTransform:
    """Simulates dermatoscope millimeter ruler tick marks near image edges."""
    def __init__(self, p=0.15):
        self.p = p

    def __call__(self, image):
        if random.random() > self.p:
            return image
        H, W = image.shape[:2]
        img_aug = image.copy()
        horizontal = random.random() < 0.5
        step = random.randint(12, 24)
        color = (random.randint(15, 50),) * 3
        if horizontal:
            y_base = random.choice([random.randint(5, int(H * 0.15)), random.randint(int(H * 0.85), H - 5)])
            for i, x in enumerate(range(10, W - 10, step)):
                tick_h = 14 if i % 5 == 0 else 7
                cv2.line(img_aug, (x, y_base), (x, min(H - 1, y_base + tick_h)), color, thickness=random.randint(1, 2))
        else:
            x_base = random.choice([random.randint(5, int(W * 0.15)), random.randint(int(W * 0.85), W - 5)])
            for i, y in enumerate(range(10, H - 10, step)):
                tick_w = 14 if i % 5 == 0 else 7
                cv2.line(img_aug, (x_base, y), (min(W - 1, x_base + tick_w), y), color, thickness=random.randint(1, 2))
        return img_aug


class SpecularGlareTransform:
    """Simulates small bright specular reflection spots from dermoscopic immersion fluid."""
    def __init__(self, p=0.15, max_spots=4):
        self.p = p
        self.max_spots = max_spots

    def __call__(self, image):
        if random.random() > self.p:
            return image
        H, W = image.shape[:2]
        img_aug = image.copy()
        for _ in range(random.randint(1, self.max_spots)):
            cx = random.randint(int(W * 0.15), int(W * 0.85))
            cy = random.randint(int(H * 0.15), int(H * 0.85))
            rx = random.randint(3, 10)
            ry = random.randint(2, 7)
            angle = random.randint(0, 180)
            overlay = img_aug.copy()
            cv2.ellipse(overlay, (cx, cy), (rx, ry), angle, 0, 360, (245, 245, 245), -1)
            alpha = random.uniform(0.4, 0.75)
            img_aug = cv2.addWeighted(overlay, alpha, img_aug, 1.0 - alpha, 0)
        return img_aug


class ImageOnlyCutoutTransform:
    """Applies small rectangular occlusions to the RGB image ONLY (keeping GT mask intact)."""
    def __init__(self, p=0.25, max_holes=4, max_size=28):
        self.p = p
        self.max_holes = max_holes
        self.max_size = max_size

    def __call__(self, image):
        if random.random() > self.p:
            return image
        H, W = image.shape[:2]
        img_aug = image.copy()
        mean_color = img_aug.mean(axis=(0, 1)).astype(np.uint8).tolist()
        for _ in range(random.randint(1, self.max_holes)):
            h = random.randint(8, self.max_size)
            w = random.randint(8, self.max_size)
            y0 = random.randint(0, max(1, H - h))
            x0 = random.randint(0, max(1, W - w))
            img_aug[y0:y0 + h, x0:x0 + w] = mean_color
        return img_aug


# ====================================================================
# 3. Canonical GDF-Net Architecture (29.50M Params)
# ====================================================================
class HierarchicalBackboneV4(nn.Module):
    def __init__(self, backbone_name=BACKBONE_NAME, pretrained=True, in_chans=3, drop_path_rate=0.2):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=pretrained,
            in_chans=in_chans,
            features_only=True,
            out_indices=(0, 1, 2, 3),
            drop_path_rate=drop_path_rate
        )
        self.channels = self.backbone.feature_info.channels()

    def get_channels(self):
        return tuple(self.channels)

    def forward(self, x):
        feats = self.backbone(x)
        return feats[0], feats[1], feats[2], feats[3]


class MultiScaleFrequencyFilter(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        self.in_channels = in_channels
        mid_channels = max(in_channels // reduction, 16)

        self.dw_conv1 = nn.Conv2d(in_channels, in_channels, kernel_size=1, groups=in_channels, bias=False)
        self.dw_conv3 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, groups=in_channels, bias=False)
        self.dw_conv5 = nn.Conv2d(in_channels, in_channels, kernel_size=5, padding=2, groups=in_channels, bias=False)

        self.pw_low = nn.Sequential(
            nn.Conv2d(in_channels * 3, in_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

        self.hf_gate = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, mid_channels, kernel_size=3, padding=1, groups=mid_channels, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, in_channels, kernel_size=1),
            nn.Sigmoid()
        )

        self.fusion = nn.Sequential(
            nn.Conv2d(in_channels * 2, in_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        low1 = self.dw_conv1(x)
        low3 = self.dw_conv3(x)
        low5 = self.dw_conv5(x)
        x_low = self.pw_low(torch.cat([low1, low3, low5], dim=1))
        x_high = x - x_low
        gate = self.hf_gate(x_low)
        x_high_filtered = x_high * gate
        out = self.fusion(torch.cat([x_low, x_high_filtered], dim=1))
        return out + x


class MaskGuidedLGAG(nn.Module):
    def __init__(self, skip_channels, gating_channels, out_channels=None):
        super().__init__()
        out_channels = out_channels or skip_channels
        self.skip_channels = skip_channels
        self.gating_channels = gating_channels

        self.g_proj = nn.Sequential(
            nn.Conv2d(gating_channels, skip_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(skip_channels)
        )

        mid_ch = max(skip_channels // 2, 16)
        self.conv1x1 = nn.Conv2d(skip_channels + 1, mid_ch, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_ch)
        self.relu = nn.ReLU(inplace=True)

        self.dw_dil1 = nn.Conv2d(mid_ch, mid_ch, kernel_size=3, padding=1, dilation=1, groups=mid_ch, bias=False)
        self.dw_dil2 = nn.Conv2d(mid_ch, mid_ch, kernel_size=3, padding=2, dilation=2, groups=mid_ch, bias=False)
        self.bn_dw = nn.BatchNorm2d(mid_ch)

        self.attn_proj = nn.Sequential(
            nn.Conv2d(mid_ch, skip_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(skip_channels),
            nn.Sigmoid()
        )

        self.out_conv = nn.Sequential(
            nn.Conv2d(skip_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, skip, gating, mask_prob=None):
        B, C, H, W = skip.shape
        g_up = F.interpolate(gating, size=(H, W), mode='bilinear', align_corners=False)
        g_proj = self.g_proj(g_up)

        if mask_prob is not None:
            m_up = F.interpolate(mask_prob, size=(H, W), mode='bilinear', align_corners=False)
        else:
            m_up = torch.zeros((B, 1, H, W), device=skip.device, dtype=skip.dtype)

        feat = skip + g_proj
        feat_with_mask = torch.cat([feat, m_up], dim=1)

        x = self.relu(self.bn1(self.conv1x1(feat_with_mask)))
        x = self.relu(self.bn_dw(self.dw_dil1(x) + self.dw_dil2(x)))
        attn = self.attn_proj(x)

        gated_skip = skip * attn
        return self.out_conv(gated_skip)


class MultiScaleConvolutionalAttentionModule(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.dw1 = nn.Conv2d(in_channels, in_channels, kernel_size=1, groups=in_channels, bias=False)
        self.dw3 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, groups=in_channels, bias=False)
        self.dw5 = nn.Conv2d(in_channels, in_channels, kernel_size=5, padding=2, groups=in_channels, bias=False)
        self.bn_dw = nn.BatchNorm2d(in_channels * 3)

        self.pw = nn.Sequential(
            nn.Conv2d(in_channels * 3, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

        mid_ch = max(out_channels // 4, 16)
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(out_channels, mid_ch, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_ch, out_channels, kernel_size=1, bias=False),
            nn.Sigmoid()
        )

        self.sa = nn.Sequential(
            nn.Conv2d(out_channels, 1, kernel_size=7, padding=3, bias=False),
            nn.Sigmoid()
        )

        self.refine = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        d1 = self.dw1(x)
        d3 = self.dw3(x)
        d5 = self.dw5(x)
        dw_cat = F.relu(self.bn_dw(torch.cat([d1, d3, d5], dim=1)), inplace=True)
        feat = self.pw(dw_cat)
        feat = feat * self.ca(feat)
        feat = feat * self.sa(feat)
        return self.refine(feat)


class CascadedMSCAMDecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.mscam = MultiScaleConvolutionalAttentionModule(in_channels + skip_channels, out_channels)
        self.head = nn.Conv2d(out_channels, 1, kernel_size=1)

    def forward(self, x, gated_skip):
        x_up = self.upsample(x)
        if x_up.shape[2:] != gated_skip.shape[2:]:
            x_up = F.interpolate(x_up, size=gated_skip.shape[2:], mode='bilinear', align_corners=False)
        fused = torch.cat([x_up, gated_skip], dim=1)
        out_feat = self.mscam(fused)
        pred_logits = self.head(out_feat)
        return out_feat, pred_logits


class GDFNet(nn.Module):
    def __init__(self, backbone_name=BACKBONE_NAME, pretrained=True, in_chans=3, decoder_channels=64, drop_path_rate=0.2):
        super().__init__()
        self.backbone_name = backbone_name
        self.decoder_channels = decoder_channels

        self.backbone = HierarchicalBackboneV4(
            backbone_name=backbone_name,
            pretrained=pretrained,
            in_chans=in_chans,
            drop_path_rate=drop_path_rate
        )
        c1, c2, c3, c4 = self.backbone.get_channels()

        self.fdf1 = MultiScaleFrequencyFilter(c1)
        self.fdf2 = MultiScaleFrequencyFilter(c2)
        self.fdf3 = MultiScaleFrequencyFilter(c3)
        self.fdf4 = MultiScaleFrequencyFilter(c4)

        dec4_ch = decoder_channels * 4
        self.dec4_proj = nn.Sequential(
            nn.Conv2d(c4, dec4_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(dec4_ch),
            nn.ReLU(inplace=True)
        )
        self.head4 = nn.Conv2d(dec4_ch, 1, kernel_size=1)

        dec3_ch = decoder_channels * 3
        dec2_ch = decoder_channels * 2
        dec1_ch = decoder_channels

        self.gate3 = MaskGuidedLGAG(skip_channels=c3, gating_channels=dec4_ch, out_channels=c3)
        self.gate2 = MaskGuidedLGAG(skip_channels=c2, gating_channels=dec3_ch, out_channels=c2)
        self.gate1 = MaskGuidedLGAG(skip_channels=c1, gating_channels=dec2_ch, out_channels=c1)

        self.dec3 = CascadedMSCAMDecoderBlock(in_channels=dec4_ch, skip_channels=c3, out_channels=dec3_ch)
        self.dec2 = CascadedMSCAMDecoderBlock(in_channels=dec3_ch, skip_channels=c2, out_channels=dec2_ch)
        self.dec1 = CascadedMSCAMDecoderBlock(in_channels=dec2_ch, skip_channels=c1, out_channels=dec1_ch)

        self.final_refine = nn.Sequential(
            nn.Conv2d(dec1_ch, dec1_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(dec1_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(dec1_ch, 1, kernel_size=1)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        f1, f2, f3, f4 = self.backbone(x)

        s1 = self.fdf1(f1)
        s2 = self.fdf2(f2)
        s3 = self.fdf3(f3)
        s4 = self.fdf4(f4)

        d4 = self.dec4_proj(s4)
        p4 = self.head4(d4)
        m4_prob = torch.sigmoid(p4)

        gated_s3 = self.gate3(s3, d4, m4_prob)
        d3, p3 = self.dec3(d4, gated_s3)
        m3_prob = torch.sigmoid(p3)

        gated_s2 = self.gate2(s2, d3, m3_prob)
        d2, p2 = self.dec2(d3, gated_s2)
        m2_prob = torch.sigmoid(p2)

        gated_s1 = self.gate1(s1, d2, m2_prob)
        d1, p1 = self.dec1(d2, gated_s1)

        p1_up = F.interpolate(p1, size=(H, W), mode='bilinear', align_corners=False)
        p2_up = F.interpolate(p2, size=(H, W), mode='bilinear', align_corners=False)
        p3_up = F.interpolate(p3, size=(H, W), mode='bilinear', align_corners=False)
        p4_up = F.interpolate(p4, size=(H, W), mode='bilinear', align_corners=False)

        d1_up = F.interpolate(d1, size=(H, W), mode='bilinear', align_corners=False)
        refined_logits = self.final_refine(d1_up)

        out_final = refined_logits + p1_up + p2_up + p3_up + p4_up

        if self.training:
            return {
                'logits': out_final,
                'p1': p1_up,
                'p2': p2_up,
                'p3': p3_up,
                'p4': p4_up
            }
        return out_final


class WarmupModelEMA:
    """EMA with dynamic warmup decay so early random decoder weights do not pollute the average."""
    def __init__(self, model, decay=0.998):
        self.ema = copy.deepcopy(model).eval()
        self.decay = decay
        self.updates = 0
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = min(self.decay, (1.0 + self.updates) / (10.0 + self.updates))
        msd = model.state_dict()
        for k, ema_v in self.ema.state_dict().items():
            model_v = msd[k].detach()
            if ema_v.dtype.is_floating_point:
                ema_v.mul_(d).add_(model_v, alpha=1.0 - d)
            else:
                ema_v.copy_(model_v)


# ====================================================================
# 4. Layer-Wise Learning Rate Decay (LLRD) Parameter Group Builder
# ====================================================================
def build_llrd_param_groups(model):
    """
    Splits GDF-Net into 5 hierarchical learning-rate groups:
      - Group 1 (Encoder Stem + Stage 0, low-level edges)   : lr = 1.5e-5
      - Group 2 (Encoder Stage 1, mid-level textures)       : lr = 2.5e-5
      - Group 3 (Encoder Stage 2, part representations)     : lr = 4.0e-5
      - Group 4 (Encoder Stage 3, semantic bottleneck)      : lr = 6.0e-5
      - Group 5 (Decoder, MS-FDF, MaskGuidedLGAG, Heads)    : lr = 2.0e-4
    """
    g_stage0, g_stage1, g_stage2, g_stage3, g_decoder = [], [], [], [], []

    for name, param in model.named_parameters():
        if name.startswith("backbone."):
            if "stages_0" in name or "stages.0" in name or "stem" in name:
                g_stage0.append(param)
            elif "stages_1" in name or "stages.1" in name:
                g_stage1.append(param)
            elif "stages_2" in name or "stages.2" in name:
                g_stage2.append(param)
            else:
                g_stage3.append(param)
        else:
            g_decoder.append(param)

    print(
        f"[+] LLRD Parameter Groups Initialized:\n"
        f"    - Stem & Stage 1 (lr=1.5e-5) : {sum(p.numel() for p in g_stage0):,} params\n"
        f"    - Stage 2        (lr=2.5e-5) : {sum(p.numel() for p in g_stage1):,} params\n"
        f"    - Stage 3        (lr=4.0e-5) : {sum(p.numel() for p in g_stage2):,} params\n"
        f"    - Stage 4        (lr=6.0e-5) : {sum(p.numel() for p in g_stage3):,} params\n"
        f"    - Decoder & FDF  (lr=2.0e-4) : {sum(p.numel() for p in g_decoder):,} params",
        flush=True
    )

    param_groups = [
        {"params": g_stage0, "lr": 1.5e-5, "name": "bb_stage0"},
        {"params": g_stage1, "lr": 2.5e-5, "name": "bb_stage1"},
        {"params": g_stage2, "lr": 4.0e-5, "name": "bb_stage2"},
        {"params": g_stage3, "lr": 6.0e-5, "name": "bb_stage3"},
        {"params": g_decoder, "lr": 2.0e-4, "name": "decoder_heads"},
    ]
    return [g for g in param_groups if len(g["params"]) > 0]


# ====================================================================
# 5. Direct Macro-Dice Optimizing Composite Loss
# ====================================================================
class StructureAndMacroDiceLoss(nn.Module):
    def __init__(self, kernel_size=31, weight_factor=5.0, dice_weight=0.5):
        super().__init__()
        self.weight_factor = weight_factor
        self.dice_weight = dice_weight
        self.avg_pool = nn.AvgPool2d(kernel_size=kernel_size, stride=1, padding=kernel_size // 2)

    def forward(self, pred_logits, targets):
        with torch.no_grad():
            local_mean = self.avg_pool(targets)
            boundary_diff = torch.abs(targets - local_mean)
            weight = 1.0 + self.weight_factor * boundary_diff

        bce = F.binary_cross_entropy_with_logits(pred_logits, targets, reduction='none')
        wbce = (weight * bce).sum(dim=(2, 3)) / (weight.sum(dim=(2, 3)) + 1e-8)

        probs = torch.sigmoid(pred_logits)
        intersection = (weight * probs * targets).sum(dim=(2, 3))
        union = (weight * (probs + targets - probs * targets)).sum(dim=(2, 3))
        wiou = 1.0 - (intersection + 1.0) / (union + 1.0)

        raw_inter = (probs * targets).sum(dim=(2, 3))
        raw_card = (probs + targets).sum(dim=(2, 3))
        soft_dice_loss = 1.0 - (2.0 * raw_inter + 1.0) / (raw_card + 1.0)

        return (wbce + wiou + self.dice_weight * soft_dice_loss).mean()


class GDFNetDiceOptCompositeLoss(nn.Module):
    def __init__(self, dice_weight=0.5):
        super().__init__()
        self.base_loss = StructureAndMacroDiceLoss(dice_weight=dice_weight)

    def forward(self, outputs, targets):
        if isinstance(outputs, dict):
            l_final = self.base_loss(outputs['logits'], targets)
            l_p1 = self.base_loss(outputs['p1'], targets)
            l_p2 = self.base_loss(outputs['p2'], targets)
            l_p3 = self.base_loss(outputs['p3'], targets)
            l_p4 = self.base_loss(outputs['p4'], targets)
            return l_final + l_p1 + 0.5 * l_p2 + 0.3 * l_p3 + 0.2 * l_p4
        return self.base_loss(outputs, targets)


# ====================================================================
# 6. Fast Local SSD Cache & Dataset Engine
# ====================================================================
IMG_SIZE = 384
TRAIN_CACHE_SIZE = 512


def prepare_fast_ssd_cache(imgs, msks, cache_dir, target_size, desc="Dataset"):
    os.makedirs(cache_dir, exist_ok=True)
    cached_imgs, cached_msks = [], []
    need_build = False
    for im_p, mk_p in zip(imgs, msks):
        base_id = os.path.splitext(os.path.basename(im_p))[0]
        c_im = os.path.join(cache_dir, f"{base_id}.jpg")
        c_mk = os.path.join(cache_dir, f"{base_id}_mask.png")
        cached_imgs.append(c_im)
        cached_msks.append(c_mk)
        if not (os.path.exists(c_im) and os.path.exists(c_mk)):
            need_build = True

    if need_build:
        print(f"[*] Building fast local SSD cache for {desc} ({len(imgs)} pairs -> {target_size}x{target_size})...", flush=True)
        t_cache = time.time()
        for im_p, mk_p, c_im, c_mk in zip(imgs, msks, cached_imgs, cached_msks):
            if not (os.path.exists(c_im) and os.path.exists(c_mk)):
                img = cv2.imread(im_p, cv2.IMREAD_COLOR)
                if img is None:
                    img = cv2.cvtColor(np.array(Image.open(im_p).convert('RGB')), cv2.COLOR_RGB2BGR)
                msk = cv2.imread(mk_p, cv2.IMREAD_GRAYSCALE)
                if msk is None:
                    msk = np.array(Image.open(mk_p).convert('L'))
                img_r = cv2.resize(img, (target_size, target_size), interpolation=cv2.INTER_AREA)
                msk_r = cv2.resize(msk, (target_size, target_size), interpolation=cv2.INTER_NEAREST)
                cv2.imwrite(c_im, img_r, [int(cv2.IMWRITE_JPEG_QUALITY), 96])
                cv2.imwrite(c_mk, msk_r)
        print(f"[+] Fast SSD cache ready for {desc} in {time.time() - t_cache:.1f}s!", flush=True)
    else:
        print(f"[+] Using existing fast SSD cache for {desc} at {cache_dir}", flush=True)
    return cached_imgs, cached_msks


class ISICDatasetV4HeavyAug(Dataset):
    """
    Loads 512x512 cached images so multi-scale training at {320, 384, 448} preserves high-res detail.
    """
    def __init__(self, imgs, msks, is_train=False, base_size=448):
        self.imgs = imgs
        self.msks = msks
        self.is_train = is_train
        self.base_size = base_size if is_train else IMG_SIZE

        self.vignetting = RandomVignettingTransform(p=0.25)
        self.hair = SyntheticHairTransform(p=0.30, max_hairs=5)
        self.ruler = SyntheticRulerMarkTransform(p=0.15)
        self.glare = SpecularGlareTransform(p=0.15)
        self.cutout = ImageOnlyCutoutTransform(p=0.25, max_holes=4, max_size=28)

        if is_train:
            self.spatial_transform = A.Compose([
                A.RandomResizedCrop(size=(self.base_size, self.base_size), scale=(0.55, 1.0), p=0.75),
                A.Resize(height=self.base_size, width=self.base_size),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.Affine(
                    scale=(0.82, 1.18),
                    translate_percent=(-0.10, 0.10),
                    rotate=(-180, 180),
                    shear=(-10, 10),
                    p=0.65
                ),
                A.OneOf([
                    A.GridDistortion(num_steps=5, distort_limit=0.25, p=1.0),
                    A.OpticalDistortion(distort_limit=0.25, p=1.0),
                    A.ElasticTransform(alpha=30, sigma=5, p=1.0),
                ], p=0.35),
            ])
            self.photo_transform = A.Compose([
                A.OneOf([
                    A.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.08, p=1.0),
                    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=25, val_shift_limit=20, p=1.0),
                    A.RandomGamma(gamma_limit=(80, 120), p=1.0),
                ], p=0.60),
                A.CLAHE(clip_limit=2.5, tile_grid_size=(8, 8), p=0.30),
                A.OneOf([
                    A.GaussianBlur(blur_limit=(3, 5), p=1.0),
                    A.GaussNoise(p=1.0),
                ], p=0.20),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
            ])
        else:
            self.eval_transform = A.Compose([
                A.Resize(height=IMG_SIZE, width=IMG_SIZE),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
            ])

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = cv2.imread(self.imgs[idx], cv2.IMREAD_COLOR)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        else:
            img = np.array(Image.open(self.imgs[idx]).convert('RGB'))

        msk = cv2.imread(self.msks[idx], cv2.IMREAD_GRAYSCALE)
        if msk is None:
            msk = np.array(Image.open(self.msks[idx]).convert('L'))
        msk = (msk > 127).astype(np.float32)

        if self.is_train:
            sp = self.spatial_transform(image=img, mask=msk)
            img, msk = sp['image'], sp['mask']
            img = self.vignetting(img)
            img = self.hair(img)
            img = self.ruler(img)
            img = self.glare(img)
            img = self.cutout(img)
            augmented = self.photo_transform(image=img, mask=msk)
        else:
            augmented = self.eval_transform(image=img, mask=msk)

        img = np.transpose(augmented['image'].astype(np.float32), (2, 0, 1))
        msk = np.expand_dims(augmented['mask'].astype(np.float32), axis=0)
        return torch.from_numpy(img).float(), torch.from_numpy(msk).float()


# ====================================================================
# 7. Comprehensive Evaluation & Calibration Engine
# ====================================================================
def compute_metrics_v4(p, t, compute_surface=True):
    p = p.astype(bool)
    t = t.astype(bool)
    tp = np.logical_and(p, t).sum()
    fp = np.logical_and(p, ~t).sum()
    fn = np.logical_and(~p, t).sum()
    tn = np.logical_and(~p, ~t).sum()

    dice = (2.0 * tp) / (2.0 * tp + fp + fn + 1e-8)
    miou = tp / (tp + fp + fn + 1e-8)
    sens = tp / (tp + fn + 1e-8)
    spec = tn / (tn + fp + 1e-8)
    prec = tp / (tp + fp + 1e-8)
    fpr  = fp / (fp + tn + 1e-8)

    hd95, asd, bf2 = 0.0, 0.0, 0.0
    if compute_surface and t.sum() > 0 and p.sum() > 0:
        bg = (t ^ binary_erosion(t)).astype(bool)
        bp = (p ^ binary_erosion(p)).astype(bool)
        if bg.sum() > 0 and bp.sum() > 0:
            dt_g = distance_transform_edt(~bg)
            dt_p = distance_transform_edt(~bp)
            dists = np.concatenate([dt_g[bp], dt_p[bg]])
            hd95 = float(np.percentile(dists, 95))
            asd  = float((dt_g[bp].mean() + dt_p[bg].mean()) / 2.0)
            prec_b = (dt_g[bp] <= 2).sum() / bp.sum()
            rec_b  = (dt_p[bg] <= 2).sum() / bg.sum()
            bf2 = float(2 * prec_b * rec_b / (prec_b + rec_b + 1e-8))

    return {
        'dice': float(dice), 'miou': float(miou), 'sensitivity': float(sens),
        'specificity': float(spec), 'precision': float(prec), 'fpr': float(fpr),
        'hd95': hd95, 'asd': asd, 'bf2': bf2,
        'tp': int(tp), 'fp': int(fp), 'fn': int(fn)
    }


@torch.no_grad()
def predict_batch_probs(model, imgs, mode="single"):
    """
    mode:
      - 'single'   : 1-view native 384x384
      - 'tta4'     : 4-view flip TTA at 384x384
      - 'tta6_ms'  : 6-view multi-scale + flip TTA ({384x384 4-view} + {448x448 2-view})
    """
    B, C, H, W = imgs.shape
    if mode == "single":
        return torch.sigmoid(model(imgs))
    elif mode == "tta4":
        p1 = torch.sigmoid(model(imgs))
        p2 = torch.flip(torch.sigmoid(model(torch.flip(imgs, dims=[3]))), dims=[3])
        p3 = torch.flip(torch.sigmoid(model(torch.flip(imgs, dims=[2]))), dims=[2])
        p4 = torch.flip(torch.sigmoid(model(torch.flip(imgs, dims=[2, 3]))), dims=[2, 3])
        return (p1 + p2 + p3 + p4) / 4.0
    elif mode == "tta6_ms":
        p1 = torch.sigmoid(model(imgs))
        p2 = torch.flip(torch.sigmoid(model(torch.flip(imgs, dims=[3]))), dims=[3])
        p3 = torch.flip(torch.sigmoid(model(torch.flip(imgs, dims=[2]))), dims=[2])
        p4 = torch.flip(torch.sigmoid(model(torch.flip(imgs, dims=[2, 3]))), dims=[2, 3])

        imgs_448 = F.interpolate(imgs, size=(448, 448), mode='bilinear', align_corners=False)
        p5_448 = torch.sigmoid(model(imgs_448))
        p5 = F.interpolate(p5_448, size=(H, W), mode='bilinear', align_corners=False)
        p6_448 = torch.flip(torch.sigmoid(model(torch.flip(imgs_448, dims=[3]))), dims=[3])
        p6 = F.interpolate(p6_448, size=(H, W), mode='bilinear', align_corners=False)
        return (p1 + p2 + p3 + p4 + p5 + p6) / 6.0
    else:
        raise ValueError(f"Unknown mode: {mode}")


@torch.no_grad()
def evaluate_model_v4(model, loader, device, mode="single", prob_thresh=0.5, compute_surface=True):
    model.eval()
    records = []
    tot_tp, tot_fp, tot_fn = 0, 0, 0

    for imgs, msks in loader:
        imgs = imgs.to(device)
        with autocast('cuda'):
            probs = predict_batch_probs(model, imgs, mode=mode)

        preds = (probs > prob_thresh).cpu().numpy()[:, 0]
        msks_np = (msks > 0.5).numpy()[:, 0]

        for i in range(len(preds)):
            m = compute_metrics_v4(preds[i], msks_np[i], compute_surface=compute_surface)
            tot_tp += m['tp']
            tot_fp += m['fp']
            tot_fn += m['fn']
            records.append(m)

    df = pd.DataFrame(records)
    global_dice = float((2.0 * tot_tp) / (2.0 * tot_tp + tot_fp + tot_fn + 1e-8))
    global_iou  = float(tot_tp / (tot_tp + tot_fp + tot_fn + 1e-8))

    summary = []
    for col in ['dice', 'miou', 'sensitivity', 'specificity', 'precision', 'fpr', 'hd95', 'asd', 'bf2']:
        summary.append({
            'Metric': col,
            'Mean': float(df[col].mean()),
            'Std': float(df[col].std()),
            'Median': float(df[col].median()),
            'Min': float(df[col].min()),
            'Max': float(df[col].max())
        })
    j_vals = df['miou'].values
    j_thresh = float(np.where(j_vals >= 0.65, j_vals, 0.0).mean())
    failed = float((j_vals < 0.65).sum())
    summary.append({'Metric': 'j_thresh', 'Mean': j_thresh, 'Std': 0.0, 'Median': j_thresh, 'Min': 0.0, 'Max': 1.0})
    summary.append({'Metric': 'failed_cases_j_lt_0.65', 'Mean': failed, 'Std': 0.0, 'Median': failed, 'Min': 0.0, 'Max': float(len(df))})

    summary_df = pd.DataFrame(summary)
    return df, summary_df, global_dice, global_iou


@torch.no_grad()
def calibrate_threshold_on_val(model, val_loader, device, mode="tta4"):
    """Finds probability threshold T in [0.30, 0.75] maximizing Macro Dice on Official Val (N=100)."""
    model.eval()
    all_probs, all_msks = [], []
    for imgs, msks in val_loader:
        imgs = imgs.to(device)
        with autocast('cuda'):
            probs = predict_batch_probs(model, imgs, mode=mode)
        all_probs.append(probs.cpu().numpy()[:, 0])
        all_msks.append((msks > 0.5).numpy()[:, 0])

    probs_np = np.concatenate(all_probs, axis=0)
    msks_np = np.concatenate(all_msks, axis=0)

    best_t, best_dice = 0.50, -1.0
    for t in np.linspace(0.30, 0.75, 46):
        preds = probs_np > t
        tp = np.logical_and(preds, msks_np).sum(axis=(1, 2))
        fp = np.logical_and(preds, ~msks_np).sum(axis=(1, 2))
        fn = np.logical_and(~preds, msks_np).sum(axis=(1, 2))
        d = ((2.0 * tp) / (2.0 * tp + fp + fn + 1e-8)).mean()
        if d > best_dice:
            best_dice = float(d)
            best_t = float(t)
    return best_t, best_dice


# ====================================================================
# 8. Main Training & Official Blind Benchmark Loop
# ====================================================================
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"[+] Active Compute Device: {device}", flush=True)
    if torch.cuda.is_available():
        print(f"[+] GPU Hardware: {torch.cuda.get_device_name(0)} (VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB)", flush=True)
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True

    OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else f"./results_{MODEL_TAG}"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    DATA_ROOT = "/kaggle/temp/isic-2018" if os.path.exists("/kaggle") else "./isic-2018"
    os.makedirs(DATA_ROOT, exist_ok=True)

    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)
    k_user = os.environ.get("KAGGLE_USERNAME", "namnguynnnn")
    k_key  = os.environ.get("KAGGLE_KEY", "KGAT_dad33ce94fb8b25d8a9ecd4ca26ce03f")
    with open(os.path.join(kaggle_dir, "kaggle.json"), "w") as f:
        json.dump({"username": k_user, "key": k_key}, f)
    os.chmod(os.path.join(kaggle_dir, "kaggle.json"), 0o600)

    def find_dir_by_name(target_name, search_roots):
        for base in search_roots:
            if not os.path.exists(base):
                continue
            for root, dirs, files in os.walk(base):
                if os.path.basename(root).lower() == target_name.lower():
                    return root
        return None

    def _unzip_all(root):
        for z in glob.glob(os.path.join(root, "*.zip")):
            print(f"[*] Extracting: {os.path.basename(z)}...", flush=True)
            try:
                with zipfile.ZipFile(z) as f:
                    f.extractall(root)
                os.remove(z)
            except Exception as e:
                print(f"[!] Warning while unzipping {z}: {e}", flush=True)

    search_roots = ["/kaggle/input", DATA_ROOT]
    train_img_dir = find_dir_by_name("ISIC2018_Task1-2_Training_Input", search_roots)
    test_gt_dir   = find_dir_by_name("ISIC2018_Task1_Test_GroundTruth", search_roots)

    if not train_img_dir:
        subprocess.run(["kaggle", "datasets", "download", "-d",
                        "tschandl/isic2018-challenge-task1-data-segmentation", "-p", DATA_ROOT], check=True)
        _unzip_all(DATA_ROOT)
    if not test_gt_dir:
        subprocess.run(["kaggle", "datasets", "download", "-d",
                        "mfahimhossen/isic2018-test-groundtruth", "-p", DATA_ROOT], check=True)
        _unzip_all(DATA_ROOT)

    val_img_dir = find_dir_by_name("ISIC2018_Task1-2_Validation_Input", search_roots)
    val_gt_dir  = find_dir_by_name("ISIC2018_Task1_Validation_GroundTruth", search_roots)
    if not val_img_dir:
        subprocess.run(["wget", "-q", "-O", f"{DATA_ROOT}/val_input.zip",
                        "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task1-2_Validation_Input.zip"], check=True)
    if not val_gt_dir:
        subprocess.run(["wget", "-q", "-O", f"{DATA_ROOT}/val_gt.zip",
                        "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task1_Validation_GroundTruth.zip"], check=True)
    _unzip_all(DATA_ROOT)

    train_img_dir = find_dir_by_name("ISIC2018_Task1-2_Training_Input", search_roots)
    train_gt_dir  = find_dir_by_name("ISIC2018_Task1_Training_GroundTruth", search_roots)
    val_img_dir   = find_dir_by_name("ISIC2018_Task1-2_Validation_Input", search_roots)
    val_gt_dir    = find_dir_by_name("ISIC2018_Task1_Validation_GroundTruth", search_roots)
    test_img_dir  = find_dir_by_name("ISIC2018_Task1-2_Test_Input", search_roots)
    test_gt_dir   = find_dir_by_name("ISIC2018_Task1_Test_GroundTruth", search_roots)

    def isic_id(path):
        base = os.path.basename(path).replace("_segmentation", "")
        return os.path.splitext(base)[0]

    def pair_data(img_dir, gt_dir):
        imgs = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
        msks = sorted(glob.glob(os.path.join(gt_dir, "*.png")))
        id_map = {isic_id(m): m for m in msks}
        paired_x, paired_y = [], []
        for im in imgs:
            imid = isic_id(im)
            if imid in id_map:
                paired_x.append(im)
                paired_y.append(id_map[imid])
        return paired_x, paired_y

    train_x_raw, train_y_raw = pair_data(train_img_dir, train_gt_dir)
    val_x_raw, val_y_raw     = pair_data(val_img_dir, val_gt_dir)
    test_x_raw, test_y_raw   = pair_data(test_img_dir, test_gt_dir)
    print(f"[+] Paired Datasets -> Train: {len(train_x_raw)} | Official Val: {len(val_x_raw)} | Blind Test: {len(test_x_raw)}", flush=True)

    CACHE_ROOT = os.path.join(DATA_ROOT, "fast_ssd_cache")
    train_x, train_y = prepare_fast_ssd_cache(train_x_raw, train_y_raw, os.path.join(CACHE_ROOT, "train_512"), TRAIN_CACHE_SIZE, desc="Train Set (512x512)")
    val_x, val_y     = prepare_fast_ssd_cache(val_x_raw, val_y_raw, os.path.join(CACHE_ROOT, "val_384"), IMG_SIZE, desc="Official Val Set (384x384)")
    test_x, test_y   = prepare_fast_ssd_cache(test_x_raw, test_y_raw, os.path.join(CACHE_ROOT, "test_384"), IMG_SIZE, desc="Official Test Set (384x384)")

    BATCH_SIZE = 12  # Safe for 448x448 multi-scale batches on 16GB P100/T4
    EPOCHS = TOTAL_EPOCHS
    NUM_WORKERS = min(4, os.cpu_count() or 2)

    train_loader = DataLoader(
        ISICDatasetV4HeavyAug(train_x, train_y, is_train=True, base_size=448),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
        pin_memory=True, persistent_workers=(NUM_WORKERS > 0), drop_last=True
    )
    val_loader = DataLoader(
        ISICDatasetV4HeavyAug(val_x, val_y, is_train=False),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
        pin_memory=True, persistent_workers=(NUM_WORKERS > 0)
    )
    test_loader = DataLoader(
        ISICDatasetV4HeavyAug(test_x, test_y, is_train=False),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
        pin_memory=True, persistent_workers=(NUM_WORKERS > 0)
    )

    model = GDFNet(backbone_name=BACKBONE_NAME, pretrained=True, drop_path_rate=0.2).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    bb_params = sum(p.numel() for p in model.backbone.parameters())
    print(f"[+] GDFNet ({MODEL_TAG}) Initialized | Backbone: {BACKBONE_NAME}", flush=True)
    print(f"[+] Channels: {model.backbone.get_channels()} | Backbone Params: {bb_params:,} | Total Params: {total_params:,}", flush=True)

    checkpoint_best = os.path.join(OUTPUT_DIR, f"GDFNet_{MODEL_TAG}_best.pth")
    checkpoint_latest = os.path.join(OUTPUT_DIR, f"GDFNet_{MODEL_TAG}_latest.pth")
    history_csv = os.path.join(OUTPUT_DIR, f"training_history_{MODEL_TAG}.csv")

    param_groups = build_llrd_param_groups(model)
    optimizer = optim.AdamW(param_groups, weight_decay=1e-4)
    ema = WarmupModelEMA(model, decay=0.998)
    criterion = GDFNetDiceOptCompositeLoss(dice_weight=0.5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=2e-6)
    scaler = GradScaler('cuda')

    best_val_dice = 0.0
    best_epoch = 0
    best_source = "init"
    history = []

    print("\n===========================================================================", flush=True)
    print(f"[*] STARTING {EPOCHS}-EPOCH PROPOSAL-3 TRAINING ({MODEL_TAG})", flush=True)
    print(f"    - Epochs 1..{FREEZE_WARMUP_EPOCHS}: Backbone Frozen Warmup @ 384x384", flush=True)
    print(f"    - Epochs {FREEZE_WARMUP_EPOCHS+1}..{MULTISCALE_END_EPOCH}: Full LLRD Unfrozen + Multi-Scale {{320, 384, 448}}", flush=True)
    print(f"    - Epochs {MULTISCALE_END_EPOCH+1}..{EPOCHS}: FixRes Fine-Tuning Locked @ 384x384", flush=True)
    print("===========================================================================", flush=True)

    multiscale_choices = [320, 384, 448]
    multiscale_weights = [0.25, 0.50, 0.25]

    for epoch in range(1, EPOCHS + 1):
        model.train()

        # 1. Handle 3-Epoch Backbone Freeze Warmup
        if epoch <= FREEZE_WARMUP_EPOCHS:
            for p in model.backbone.parameters():
                p.requires_grad_(False)
            stage_desc = "WARMUP-FROZEN (384)"
        else:
            if epoch == FREEZE_WARMUP_EPOCHS + 1:
                print(f"[*] Epoch {epoch}: Unfreezing ConvFormer-S18 Backbone with 5-Group LLRD & Resetting EMA Warmup!", flush=True)
                for p in model.backbone.parameters():
                    p.requires_grad_(True)
                ema = WarmupModelEMA(model, decay=0.998)
            stage_desc = "LLRD+MS{320,384,448}" if epoch <= MULTISCALE_END_EPOCH else "LLRD+FIXRES(384)"

        running_loss = 0.0
        t0 = time.time()

        for imgs, msks in train_loader:
            imgs = imgs.to(device, non_blocking=True)
            msks = msks.to(device, non_blocking=True)

            # 2. Dynamic Multi-Scale Batch Interpolation
            if epoch <= FREEZE_WARMUP_EPOCHS or epoch > MULTISCALE_END_EPOCH:
                target_s = 384
            else:
                target_s = random.choices(multiscale_choices, weights=multiscale_weights, k=1)[0]

            if imgs.shape[2] != target_s:
                imgs = F.interpolate(imgs, size=(target_s, target_s), mode='bilinear', align_corners=False)
                msks = F.interpolate(msks, size=(target_s, target_s), mode='nearest')

            optimizer.zero_grad(set_to_none=True)
            with autocast('cuda'):
                outs = model(imgs)
                loss = criterion(outs, msks)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            ema.update(model)
            running_loss += loss.item()

        scheduler.step()
        elapsed = time.time() - t0
        epoch_loss = running_loss / len(train_loader)
        lrs = scheduler.get_last_lr()
        lr_bb_min, lr_dec = lrs[0], lrs[-1]

        _, val_sum_raw, val_glob_raw, _ = evaluate_model_v4(model, val_loader, device, mode="single", compute_surface=False)
        val_dice_raw = float(val_sum_raw.loc[val_sum_raw['Metric'] == 'dice', 'Mean'].values[0])

        _, val_sum_ema, val_glob_ema, _ = evaluate_model_v4(ema.ema, val_loader, device, mode="single", compute_surface=False)
        val_dice_ema = float(val_sum_ema.loc[val_sum_ema['Metric'] == 'dice', 'Mean'].values[0])

        epoch_best_dice = max(val_dice_raw, val_dice_ema)
        epoch_best_src = "EMA" if val_dice_ema >= val_dice_raw else "RAW"

        history.append({
            'epoch': epoch,
            'stage': stage_desc,
            'loss': epoch_loss,
            'lr_bb_low': lr_bb_min,
            'lr_decoder': lr_dec,
            'val_macro_dice_raw': val_dice_raw,
            'val_global_dice_raw': val_glob_raw,
            'val_macro_dice_ema': val_dice_ema,
            'val_global_dice_ema': val_glob_ema,
            'elapsed_sec': elapsed
        })
        pd.DataFrame(history).to_csv(history_csv, index=False)

        print(
            f"Ep {epoch:02d}/{EPOCHS} [{stage_desc} | {elapsed:.1f}s] | Loss: {epoch_loss:.4f} "
            f"| LR(bb/dec): {lr_bb_min:.6f}/{lr_dec:.6f} | Val Raw: {val_dice_raw:.4f} "
            f"| Val EMA: {val_dice_ema:.4f} | Best: {max(best_val_dice, epoch_best_dice):.4f}",
            flush=True
        )

        if epoch_best_dice > best_val_dice:
            best_val_dice = epoch_best_dice
            best_epoch = epoch
            best_source = epoch_best_src
            best_state = ema.ema.state_dict() if epoch_best_src == "EMA" else model.state_dict()
            torch.save(best_state, checkpoint_best)
            print(f"[*] New Best Model ({best_source}) Saved at Epoch {epoch} (Official Val Macro Dice: {best_val_dice:.4f})", flush=True)

        if epoch == EPOCHS or (epoch % 10 == 0):
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'ema_state': ema.ema.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'scaler_state': scaler.state_dict(),
                'best_val_dice': best_val_dice,
                'best_epoch': best_epoch,
                'best_source': best_source
            }, checkpoint_latest)

        if (time.time() - SCRIPT_START_TIME) > MAX_RUNTIME_SECONDS:
            print("[!] Wall-clock safety guard reached. Proceeding to 1,000 Blind Test Evaluation...", flush=True)
            break

    print("\n===========================================================================", flush=True)
    print(f"[*] STARTING OFFICIAL 1,000 BLIND TEST EVALUATION ({MODEL_TAG})", flush=True)
    print(f"    Best Checkpoint: Epoch {best_epoch} [{best_source}] (Val Single-View Dice: {best_val_dice:.4f})", flush=True)
    print("===========================================================================", flush=True)

    if os.path.exists(checkpoint_best):
        model.load_state_dict(torch.load(checkpoint_best, map_location=device, weights_only=False))

    # 1. Single-View (T = 0.50, No TTA)
    sv_df, sv_summary, sv_glob_dice, sv_glob_iou = evaluate_model_v4(
        model, test_loader, device, mode="single", prob_thresh=0.50, compute_surface=True
    )
    sv_summary.to_csv(os.path.join(OUTPUT_DIR, f"benchmark_summary_{MODEL_TAG}_SingleView.csv"), index=False)
    sv_df.to_csv(os.path.join(OUTPUT_DIR, f"benchmark_raw_{MODEL_TAG}_SingleView.csv"), index=False)
    print(f"\n================ 1. {MODEL_TAG} BLIND TEST (SINGLE VIEW, T=0.50) ================", flush=True)
    print(sv_summary.to_string(index=False), flush=True)
    print(f"[+] Single-View Global Micro Dice: {sv_glob_dice:.4f} | IoU: {sv_glob_iou:.4f}", flush=True)

    # 2. Calibrated Single-View (T = T_opt_sv on Official Val, No TTA)
    opt_t_sv, val_sv_cal_dice = calibrate_threshold_on_val(model, val_loader, device, mode="single")
    print(f"\n[*] Calibrated Single-View Threshold on 100 Official Val: T_opt_sv = {opt_t_sv:.3f} (Val SV Dice: {val_sv_cal_dice:.4f})", flush=True)
    sv_cal_df, sv_cal_summary, sv_cal_glob_dice, sv_cal_glob_iou = evaluate_model_v4(
        model, test_loader, device, mode="single", prob_thresh=opt_t_sv, compute_surface=True
    )
    sv_cal_summary.to_csv(os.path.join(OUTPUT_DIR, f"benchmark_summary_{MODEL_TAG}_SingleView_Calibrated.csv"), index=False)
    sv_cal_df.to_csv(os.path.join(OUTPUT_DIR, f"benchmark_raw_{MODEL_TAG}_SingleView_Calibrated.csv"), index=False)
    print(f"\n================ 2. {MODEL_TAG} BLIND TEST (CALIBRATED SINGLE VIEW, T={opt_t_sv:.2f}) ================", flush=True)
    print(sv_cal_summary.to_string(index=False), flush=True)
    print(f"[+] Calibrated Single-View Global Micro Dice: {sv_cal_glob_dice:.4f} | IoU: {sv_cal_glob_iou:.4f}", flush=True)

    # 3. 4-View Flip TTA (T = 0.50)
    tta_df, tta_summary, tta_glob_dice, tta_glob_iou = evaluate_model_v4(
        model, test_loader, device, mode="tta4", prob_thresh=0.50, compute_surface=True
    )
    tta_summary.to_csv(os.path.join(OUTPUT_DIR, f"benchmark_summary_{MODEL_TAG}_TTA.csv"), index=False)
    tta_df.to_csv(os.path.join(OUTPUT_DIR, f"benchmark_raw_{MODEL_TAG}_TTA.csv"), index=False)
    print(f"\n================ 3. {MODEL_TAG} BLIND TEST (4-VIEW TTA, T=0.50) ================", flush=True)
    print(tta_summary.to_string(index=False), flush=True)
    print(f"[+] 4-View TTA Global Micro Dice: {tta_glob_dice:.4f} | IoU: {tta_glob_iou:.4f}", flush=True)

    # 4. Calibrated 4-View Flip TTA (T = T_opt_tta on Official Val)
    opt_t_tta, val_tta_cal_dice = calibrate_threshold_on_val(model, val_loader, device, mode="tta4")
    print(f"\n[*] Calibrated 4-View TTA Threshold on 100 Official Val: T_opt_tta = {opt_t_tta:.3f} (Val TTA Dice: {val_tta_cal_dice:.4f})", flush=True)
    cal_df, cal_summary, cal_glob_dice, cal_glob_iou = evaluate_model_v4(
        model, test_loader, device, mode="tta4", prob_thresh=opt_t_tta, compute_surface=True
    )
    cal_summary.to_csv(os.path.join(OUTPUT_DIR, f"benchmark_summary_{MODEL_TAG}_TTA_Calibrated.csv"), index=False)
    cal_df.to_csv(os.path.join(OUTPUT_DIR, f"benchmark_raw_{MODEL_TAG}_TTA_Calibrated.csv"), index=False)
    print(f"\n================ 4. {MODEL_TAG} BLIND TEST (4-VIEW TTA, CALIBRATED T={opt_t_tta:.2f}) ================", flush=True)
    print(cal_summary.to_string(index=False), flush=True)
    print(f"[+] Calibrated 4-View TTA Global Micro Dice: {cal_glob_dice:.4f} | IoU: {cal_glob_iou:.4f}", flush=True)

    # 5. Multi-Scale 6-View TTA ({384, 448} + Flips, Calibrated on Official Val)
    opt_t_ms, val_ms_cal_dice = calibrate_threshold_on_val(model, val_loader, device, mode="tta6_ms")
    print(f"\n[*] Calibrated 6-View Multi-Scale TTA Threshold on 100 Official Val: T_opt_ms = {opt_t_ms:.3f} (Val MS-TTA Dice: {val_ms_cal_dice:.4f})", flush=True)
    ms_df, ms_summary, ms_glob_dice, ms_glob_iou = evaluate_model_v4(
        model, test_loader, device, mode="tta6_ms", prob_thresh=opt_t_ms, compute_surface=True
    )
    ms_summary.to_csv(os.path.join(OUTPUT_DIR, f"benchmark_summary_{MODEL_TAG}_MSTTA_Calibrated.csv"), index=False)
    ms_df.to_csv(os.path.join(OUTPUT_DIR, f"benchmark_raw_{MODEL_TAG}_MSTTA_Calibrated.csv"), index=False)
    print(f"\n================ 5. {MODEL_TAG} BLIND TEST (6-VIEW MULTI-SCALE TTA, CALIBRATED T={opt_t_ms:.2f}) ================", flush=True)
    print(ms_summary.to_string(index=False), flush=True)
    print(f"[+] Calibrated 6-View Multi-Scale TTA Global Micro Dice: {ms_glob_dice:.4f} | IoU: {ms_glob_iou:.4f}", flush=True)

    print("\n[+] GDF-Net Teacher M2 (Stage-Wise LLRD + Multi-Scale) Training & 5-Mode Official ISIC-2018 Blind Evaluation Complete!", flush=True)


if __name__ == "__main__":
    main()


[+] Active Compute Device: cuda
[+] GPU Hardware: Tesla T4 (VRAM: 15.64 GB)
[+] Paired Datasets -> Train: 2594 | Official Val: 100 | Blind Test: 1000
[*] Building fast local SSD cache for Train Set (512x512) (2594 pairs -> 512x512)...
[+] Fast SSD cache ready for Train Set (512x512) in 662.3s!
[*] Building fast local SSD cache for Official Val Set (384x384) (100 pairs -> 384x384)...
[+] Fast SSD cache ready for Official Val Set (384x384) in 13.2s!
[*] Building fast local SSD cache for Official Test Set (384x384) (1000 pairs -> 384x384)...
[+] Fast SSD cache ready for Official Test Set (384x384) in 197.4s!
[+] GDFNet (ConvFormerS18_LLRD_MS) Initialized | Backbone: convformer_s18.sail_in22k_ft_in1k_384
[+] Channels: (64, 128, 320, 512) | Backbone Params: 23,669,704 | Total Params: 29,503,181
[+] LLRD Parameter Groups Initialized:
    - Stem & Stage 1 (lr=1.5e-5) : 176,204 params
    - Stage 2        (lr=2.5e-5) : 702,156 params
    - Stage 3        (lr=4.0e-5) : 11,722,084 params
    - S